In [0]:
# =============================================================================
# Notebook: 01_infrastructure_setup.ipynb
# Purpose:  One-time provisioning of all Unity Catalog infrastructure
#           across Bronze, Silver, and Gold layers.
# Run:      Execute once before running any pipeline for the first time.
#           Safe to rerun — all commands use IF NOT EXISTS.
# =============================================================================

print("Provisioning full Medallion Architecture infrastructure...")

# --- BRONZE LAYER ---
spark.sql("CREATE CATALOG IF NOT EXISTS bronze_dev")
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze_dev.api_sports")
spark.sql("CREATE VOLUME IF NOT EXISTS bronze_dev.api_sports.landing_zone")
print("✓ Bronze infrastructure ready: bronze_dev.api_sports.landing_zone")

# --- SILVER LAYER ---
spark.sql("CREATE CATALOG IF NOT EXISTS silver_dev")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver_dev.football_analytics")
print("✓ Silver infrastructure ready: silver_dev.football_analytics")

# --- GOLD LAYER ---
spark.sql("CREATE CATALOG IF NOT EXISTS gold_dev")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold_dev.football_analytics")
print("✓ Gold infrastructure ready: gold_dev.football_analytics")

print("\nAll Medallion Architecture infrastructure provisioned successfully.")
print("You may now run the Bronze ingestion pipeline followed by the LakeFlow pipeline.")

In [0]:
# Check Silver — confirm 607 clean active players
display(spark.sql("SELECT COUNT(*) as total_players, COUNT(DISTINCT nationality) as nationalities, COUNT(DISTINCT team_name) as teams FROM gold_dev.football_analytics.players_cleaned"))

# Check Colombian players — should show 6 players
display(spark.sql("SELECT player_name, team_name, position, goals_total, assists_total, minutes_played, rating FROM gold_dev.football_analytics.player_performance ORDER BY goals_total DESC"))

# Check ranking — confirm goal contributions are correct
display(spark.sql("SELECT performance_rank, player_name, team_name, position, goals_total, assists_total, goal_contributions FROM gold_dev.football_analytics.player_ranking ORDER BY performance_rank"))

# Check team contribution
display(spark.sql("SELECT team_name, total_players, total_goals, total_assists, total_minutes FROM gold_dev.football_analytics.team_nationality_contribution ORDER BY total_goals DESC"))

# Check position breakdown
display(spark.sql("SELECT position, total_players, total_goals, total_assists, avg_minutes_played FROM gold_dev.football_analytics.position_performance_breakdown"))

# Check league summary — should be exactly 1 row
display(spark.sql("SELECT * FROM gold_dev.football_analytics.league_nationality_summary"))

In [0]:
display(spark.sql("""
    SELECT 
        avg_player_rating,
        avg_pass_accuracy,
        avg_duel_win_rate,
        total_yellow_cards,
        total_red_cards
    FROM gold_dev.football_analytics.league_nationality_summary
"""))